# Predictive Modeling for Employee Promotion

## Phase 1: Frame the Problem and Look at the Big Picture

### Objective:
Our goal is to predict which employees are likely to be promoted based on performance metrics and demographics. 
This is a **binary classification** task. We are looking for the 'promoted' target variable.


### 1. Environment Setup
We are initializing our workspace by importing the core libraries:
* **Pandas**: For structured data manipulation.
* **Numpy**: For linear algebra and numerical processing.
* **Matplotlib/Seaborn**: For the "Data Discovery" visualization phase.
* **imbalanced-learn**: For addressing any "class imbalance" problem.

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for all future charts
sns.set_theme(style="whitegrid")
print("Libraries imported. The project environment is ready.")

# Phase 2: Data Acquisition

### 1. Loading the Dataset
Following the project workspace setup, we now move to the "Get the Data" phase. We are loading the employee promotion dataset to begin our initial discovery.

**Data Source:** Kaggle

In [ ]:
import pandas as pd

try:
    df = pd.read_csv('../Data/employee_promotion_prediction.csv')
    print(" SUCCESS: Data is loaded.")
    print(f"Dataset Size: {df.shape}")
    display(df.head())
except Exception as e:
    print(f" Could not load data. Error: {e}")

### 2. Data Inspection and Integrity Check
In this step, we perform a deep dive into the dataset's metadata. 
we need to identify:

* **Data Types**: Are numerical values stored as `int64` or `float64`?

* **Missing Values**: Which features require imputation?

* **Statistical Overview**: Are there any obvious outliers in the numerical data?

In [ ]:
# Check data types and non-null counts
print("--- Dataset Information ---")
display(df.info())

In [ ]:
# Descriptive Statistics
print("\n--- Numerical Summary ---")
display(df.describe())

## Phase 3: Explore the Data

### 1. Target Variable Analysis
We are predicting `promoted`. We need to visualize the distribution of this class to confirm the level of **class imbalance**. 

In real-world HR data, the number of employees promoted is usually much smaller than those who are not.

In [ ]:
import matplotlib.pyplot as plt

# 1. Use the correct column name from your df.columns list
target_col = 'promoted' 

# 2. Extract counts
counts = df[target_col].value_counts()
labels = ['Not Promoted (0)', 'Promoted (1)']
values = [counts.get(0, 0), counts.get(1, 0)]

# 3. Create the bar chart
plt.figure(figsize=(8, 6))
plt.bar(labels, values, color=['#34495e', '#27ae60'])

# 4. Add formatting  
plt.title('Distribution of Promotions (Target Variable)', fontsize=14)
plt.ylabel('Number of Employees', fontsize=12)

# Add value labels on top of bars
for i, v in enumerate(values):
    plt.text(i, v + (max(values)*0.01), str(v), ha='center', fontweight='bold')

plt.show()

# 5. Print the proportions
print("Target Proportions (%):")
print(df[target_col].value_counts(normalize=True) * 100)

### 2. Attribute Study: Distribution and Characteristics
We then visualize the distributions of all numerical attributes. 
This allows us to identify:
1. **Distribution Types**
2. **Attribute Scales**: The range of values for each feature.
3. **Outliers and Noise**: Values that sit far outside the norm.

In [ ]:
import matplotlib.pyplot as plt

# Plotting histograms for all numerical attributes
# We use 50 bins to see the detail in the distributions
df.hist(bins=50, figsize=(20,15))
plt.tight_layout()
plt.show()

### 2.1  Identifying Skewed Attributes
To prepare our features for the Gaussian Naive Bayes model, we evaluate the skewness of all numerical attributes. 

We focus on features with a skewness score greater than 0.75, which we will target for a log transformation to normalize their distributions.

In [ ]:
# 1. Select all numerical columns from the full dataframe
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# 2. Calculate skewness for all of them
skew_values = df[numerical_cols].skew().sort_values(ascending=False)

# 3. Create a DataFrame for readability
skew_df = pd.DataFrame({'Skewness': skew_values})

# 4. Filter for high skewness (threshold > 0.75)
skewed_attributes = skew_df[skew_df['Skewness'] > 0.75].index.tolist()

print("--- Skewness Report (All Numerical Variables) ---")
print(skew_df)
print(f"\nHighly Skewed Attributes to Consider for Transformation: {skewed_attributes}")

### 3. Correlation Analysis 
The heatmap below displays the strength of these relationships. 

We specifically look for variables that have a higher absolute correlation with `promoted`, as these are the most "promising" features for our machine learning models.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculate the correlation matrix
corr_matrix = df.select_dtypes(include=[np.number]).corr()

# 2. Set a significantly larger figure size to prevent label overlap
plt.figure(figsize=(25, 20))

# 3. Create a mask for the upper triangle (redundant data)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 4. Generate the Heatmap
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True,          # Show the numerical values
    fmt=".2f",           # Limit to 2 decimal places to save space
    cmap='coolwarm',     # Red for positive, Blue for negative
    linewidths=0.5,      # Add a small gap between squares
    annot_kws={"size": 9}, # Make the font small enough to fit inside squares
    cbar_kws={"shrink": .8}
)

# Rotate labels for better readability
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

plt.title(' Correlation Heatmap: Identifying Promising Features', fontsize=18)
plt.show()

# 5. Output raw values for the target specifically
print("--- Standard Correlation Coefficients for 'promoted' ---")
corr_matrix['promoted'].reindex(corr_matrix['promoted'].abs().sort_values(ascending=False).index)



### 4. Outlier Detection and Data Quality Assurance 
Before finalizing our dataset for training, we must identify and assess extreme values (outliers) that deviate significantly from the expected range. 

Such values can introduce noise and disproportionately influence model parameters, particularly in sensitive architectures like Neural Networks.

* **Methodology:** We utilize the Interquartile Range (IQR) method. This approach defines outliers as observations falling outside the bounds:
             
             Lower Bound = Q1 - 1.5 \times IQR
             Upper Bound = Q3 + 1.5 \times IQR

* **Objective**: By quantifying outliers per column, we can determine whether to drop these records, cap them (winsorization), or investigate them as legitimate (albeit rare) edge cases within our employee dataset.

In [ ]:
# 1. Identify numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# 2. Calculate IQR for all numerical columns
Q1 = df[numerical_cols].quantile(0.25)
Q3 = df[numerical_cols].quantile(0.75)
IQR = Q3 - Q1

# 3. Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. Count outliers per column
outliers_count = ((df[numerical_cols] < lower_bound) | (df[numerical_cols] > upper_bound)).sum()
print("Number of outliers per column:")
print(outliers_count[outliers_count > 0])
print(f"\nTotal count of outliers across all columns: {outliers_count.sum()}")

# Phase 3: Prepare the Data
We now prepare the data for our three models (Naive Bayes, Random Forest, and Neural Network).

**Sub-steps:**
* **Data Cleaning**
* **Feature Selection** 
* **Handling Text/Categorical Attributes** 
* **Feature Scaling** 


### 2. Dataset Partitioning
 We isolated the target variable promoted, ensuring it is excluded from the feature set to prevent data leakage.
 * **Goal:** To establish a clean, standardized dataset that serves as a consistent foundation for all subsequent modeling architectures.

 We then proceeded to partition our data into dedicated training and testing sets. This ensures our models are evaluated on independent, unseen data, providing an unbiased assessment of their predictive performance.

* **Methodology:** We utilize a stratified split, maintaining the class proportions of our target variable across both training and testing sets.

* **Objective:**

   * **Training Set:** Provides the model with the necessary patterns and signals to learn associations between employee attributes and promotion outcomes.

   * **Testing Set:** Acts as a "hold-out" evaluation suite, allowing us to validate the model's performance on new, independent data to detect overfitting.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. First, we separate features (X) from target (y)
X = df.drop('promoted', axis=1) 
y = df['promoted']

# 2. Perform the split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

### 1. Feature Redundancy and Multicollinearity
We performed a redundancy check to identify and remove highly correlated features (threshold > 0.85$).
* **Why:** High multicollinearity can inflate variance and degrade the interpretability of model coefficients, especially in linear-based approaches.
* **Mechanism:** By analyzing the upper triangle of the correlation matrix, we systematically pruned redundant inputs, ensuring each feature contributes unique variance to the model.


In [ ]:
# 1. Calculate correlation matrix using ONLY X_train
corr_matrix = X_train.corr(numeric_only=True).abs()

# 2. Identify highly correlated features
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

# 3. Add 'employee_id' if present in the training features
if 'employee_id' in X_train.columns:
    to_drop.append('employee_id')

# 4. Drop columns from BOTH training and testing sets
X_train_cleaned = X_train.drop(columns=to_drop)
X_test_cleaned = X_test.drop(columns=to_drop)

print(f"Features dropped based on Training Set: {to_drop}")
print(f"New X_train shape: {X_train_cleaned.shape}")
print(f"New X_test shape: {X_test_cleaned.shape}")

### 3. Outlier Treatment: Winsorization
Rather than removing records containing extreme values ,which can lead to a loss of valuable information and introduce selection bias, we utilize Winsorization to cap outliers at specified percentile thresholds.

* **Methodology:** We utilize the Interquartile Range (IQR) method, where any value falling below the lower bound or above the upper bound is "clipped" to these limits:
* **Objective:** This process reduces the influence of extreme anomalies while preserving the integrity of the full dataset (N=100,000$). By capping these values, we prevent them from disproportionately driving the gradient updates in Neural Networks or biasing the splits in Decision Tree-based models.

In [ ]:
# 1. Identify numerical columns available in our training set
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# 2. Calculate the bounds ONLY from X_train
Q1 = X_train[numerical_cols].quantile(0.25)
Q3 = X_train[numerical_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Apply capping to BOTH X_train and X_test
for col in numerical_cols:
    X_train[col] = X_train[col].clip(lower=lower_bound[col], upper=upper_bound[col])
    X_test[col] = X_test[col].clip(lower=lower_bound[col], upper=upper_bound[col])

print("Outliers capped using training-set bounds.")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

### 4.1 Categorical Feature Analysis

Before applying encoding techniques (such as One-Hot or Ordinal Encoding), we must audit our dataset to identify categorical variables and determine their cardinality.


In [ ]:
# 1. Identify categorical columns in the training set
cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

print("Categorical columns found in Training Set:")
for col in cat_cols:
    # 2. Inspecting only the Training set
    unique_count = X_train[col].nunique()
    print(f"- {col}: {unique_count} unique categories")

# 3. Display the head of the categorical training data
display(X_train[cat_cols].head())

### 4.2 Numerical Encoding
Following our audit, we translate these human-readable labels into numerical formats that our algorithms can interpret:

* **Ordinal Mapping:** For hierarchical features like education_level and city_tier, we apply manual mapping.
This preserves the logical progression inherent in the data, which is a critical signal for our promotion model.

* **One-Hot Encoding:** For nominal features (e.g., department, gender), we use One-Hot Encoding to create binary indicators.

In [ ]:
# 1. Define mappings
education_mapping = {"Bachelor": 1, "Master": 2, "PhD": 3}
city_mapping = {"Tier3": 0, "Tier2": 1, "Tier1": 2}

# 2. Apply to Training set
X_train['education_level'] = X_train['education_level'].map(education_mapping)
X_train['city_tier'] = X_train['city_tier'].map(city_mapping)

# 3. Apply to Testing set
X_test['education_level'] = X_test['education_level'].map(education_mapping)
X_test['city_tier'] = X_test['city_tier'].map(city_mapping)

# 4. Encoding Function
def encode_all_objects(df_train, df_test):
    obj_cols = df_train.select_dtypes(include=['object', 'category', 'str']).columns
    # Get_dummies on the whole set
    df_train_enc = pd.get_dummies(df_train, columns=obj_cols, drop_first=True, dtype=int)
    df_test_enc = pd.get_dummies(df_test, columns=obj_cols, drop_first=True, dtype=int)
    # Align
    df_test_enc = df_test_enc.reindex(columns=df_train_enc.columns, fill_value=0)
    return df_train_enc, df_test_enc

# 5. OVERWRITE the variables so they now hold the encoded numeric data
X_train, X_test = encode_all_objects(X_train, X_test)

# 6. Final safety: Ensure no NaNs exist (caused by mapping errors or missing data)
X_train = X_train.fillna(0)
y_train = y_train.fillna(0) # Ensure y_train also has no NaNs

print(f"X_train shape: {X_train_final.shape}")
print(f"X_test shape: {X_test_final.shape}")

### 5. Feature Transformation and Standardization
To improve model convergence and performance, we apply a two-step preprocessing pipeline to our numerical features:

* **Log Transformation:** We apply np.log1p to features identified as highly skewed (e.g., salary, bonus_last_year). This compresses long-tailed distributions and reduces the impact of extreme outliers.

* **Feature Standardization:** Finally, we use StandardScaler to transform all features to a common scale (mean = 0, standard deviation = 1). This ensures that no single feature dominates the model training process based purely on its magnitude.

In [ ]:
constant_cols = [col for col in X_train_transformed.columns if X_train_transformed[col].nunique() <= 1]
print("Constant columns:", constant_cols)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# Drop constant columns
cols_to_drop = ['education_level', 'city_tier']

X_train_final.drop(columns=cols_to_drop, inplace=True, errors='ignore')
X_test_final.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print("Dropped constant columns:", constant_cols)

# 2. Identify skewed features (these are constant, they don't depend on data)
skewed_features = ['salary', 'years_at_company', 'bonus_last_year', 'years_in_current_role']

# 3. Apply Log Transformation
# It is safe to apply to both sets independently.
X_train_transformed = X_train_final.copy()
X_test_transformed = X_test_final.copy()

for col in skewed_features:
    X_train_transformed[col] = np.log1p(X_train_transformed[col])
    X_test_transformed[col] = np.log1p(X_test_transformed[col])

# 4. Apply Standardization (The "Learner" Pattern)
scaler = StandardScaler()

# Fit ONLY on the training set to calculate mean and std
scaler.fit(X_train_transformed)

# Transform BOTH sets using the training parameters
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train_transformed), 
    columns=X_train_transformed.columns, 
    index=X_train_transformed.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_transformed), 
    columns=X_test_transformed.columns, 
    index=X_test_transformed.index
)

print("Preprocessing Complete: Log Transform + Scaling strictly applied to Train then Test.")

### 5.1 Post-Preprocessing Distribution Audit
After applying transformations, we perform a visual check to evaluate the effectiveness of our data preparation.

* **Methodology:** We generate histograms for all numerical features to visualize the resulting distributions.

* **Objective:**
Detect any remaining anomalies or irregularities that might indicate improper scaling or data leakage.

In [ ]:
import matplotlib.pyplot as plt

# 1. Visualize Training Data
print("Visualizing Training Set Distributions:")
X_train_scaled.hist(bins=50, figsize=(20, 15))
plt.suptitle("Training Set Feature Distributions (Scaled)")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 2. Visualize Testing Data
print("Visualizing Testing Set Distributions:")
X_test_scaled.hist(bins=50, figsize=(20, 15))
plt.suptitle("Testing Set Feature Distributions (Scaled)")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### 5.2 . Feature Binning: Discretizing Spiky Distributions
As a final refinement step, we have addressed the "spiky" distributions—where data is heavily concentrated at discrete intervals—by converting these numerical metrics into structured, categorical bins.

* **Methodology:** We defined a binning_config dictionary mapping continuous variables (such as years_at_company or mentoring_sessions) to specific, meaningful intervals.

In [ ]:
# 1. Define your binning configuration (this is static logic)
binning_config = {
    'years_at_company': {'bins': [0, 1, 3, 5, 10, 50], 'labels': ['New', 'Junior', 'Mid', 'Senior', 'Veteran']},
    'years_in_current_role': {'bins': [0, 1, 3, 5, 10, 50], 'labels': ['New', 'Short', 'Mid', 'Established', 'Senior']},
    'years_since_last_promotion': {'bins': [0, 1, 3, 5, 10, 50], 'labels': ['Recent', 'Short', 'Mid', 'Long', 'Very Long']},
    'mentoring_sessions': {'bins': [-1, 0, 2, 5, 10, 100], 'labels': ['None', 'Low', 'Medium', 'High', 'Expert']},
    'late_days': {'bins': [-1, 0, 5, 10, 20, 365], 'labels': ['Perfect', 'Good', 'Average', 'High', 'Frequent']},
    'age': {'bins': [17, 25, 35, 45, 55, 100], 'labels': ['GenZ', 'Young', 'Mid', 'Senior', 'Veteran']},
    'team_size': {'bins': [0, 5, 10, 20, 50, 1000], 'labels': ['Small', 'Medium', 'Large', 'Enterprise', 'Huge']},
    'projects_completed': {'bins': [-1, 0, 5, 10, 15, 100], 'labels': ['None', 'Low', 'Medium', 'High', 'Expert']},
    'certifications_count': {'bins': [-1, 0, 2, 4, 6, 20], 'labels': ['None', 'Low', 'Medium', 'High', 'Expert']},
    'cross_department_projects': {'bins': [-1, 0, 1, 2, 3, 20], 'labels': ['None', 'Low', 'Medium', 'High', 'Expert']}
}

# 2. Function to apply binning and dummy encoding safely
def process_bins_and_dummies(df, config):
    df_processed = df.copy()
    for col, cfg in config.items():
        if col in df_processed.columns:
            # Create categorical groups and force them to be of 'category' type
            df_processed[col + '_group'] = pd.cut(
                df_processed[col], 
                bins=cfg['bins'], 
                labels=cfg['labels']
            ).astype('category') # Explicitly setting type
            
            # Drop original spiky column
            df_processed = df_processed.drop(col, axis=1)
    
    # 3. Get dummies
    df_dummies = pd.get_dummies(df_processed, drop_first=True, dtype=int)
    
    # Final check: Ensure absolutely everything is numeric
    return df_dummies.apply(pd.to_numeric)

# Now applying this to out split data
X_train_binned = process_bins_and_dummies(X_train, binning_config)
X_test_binned = process_bins_and_dummies(X_test, binning_config)

# 4. Align the columns
X_test_binned = X_test_binned.reindex(columns=X_train_binned.columns, fill_value=0)

# Verification
non_numeric = X_train_binned.select_dtypes(include=['object', 'category']).columns
print(f"Columns remaining as non-numeric: {non_numeric}")

# Phase 4: Select and Train Models

we now enter the modeling phase. Our goal is to train multiple models, evaluate their performance, and select the most promising one.


### 1. Addressing Class Imbalance: SMOTE
We apply SMOTE to synthesize new examples for the minority class, ensuring the model learns to identify both outcomes effectively.

* **Methodology:** SMOTE generates synthetic samples by interpolating between existing minority class instances in the feature space, rather than simply duplicating data.

* **Objective:**

   * **Balanced Learning:** Prevents the model from developing a "majority class bias" where it ignores the minority outcome to achieve high accuracy.

   * **Robust Generalization:** Improves the model’s sensitivity (Recall) for the minority class, which is often the primary goal in business performance modeling.

In [ ]:
from imblearn.over_sampling import SMOTE

# 1. Initialize SMOTE
# k_neighbors=5 is standard; adjust if you have very few minority samples
smote = SMOTE(random_state=42)

# 2. Apply ONLY to the Training set
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Original training shape: {X_train.shape}")
print(f"Resampled training shape: {X_train_resampled.shape}")

### 2. Utility Functions for Model Evaluation
After training our models on the balanced training set, we evaluate their predictive performance on the unseen testing set. This step is critical for determining which model best generalizes to new data and aligns with our business objectives.

We define a custom **utility function** to quantify the business value of our predictions. This function assigns specific weights to True Positives (correctly identified promotions) and False Positives (misallocated resources), allowing us to select the model that maximizes actual organizational impact rather than just statistical accuracy.

* **Performance Metrics** We utilize a comprehensive suite of metrics to ensure a holistic evaluation:

* **Accuracy:** The overall correctness of our predictions.

* **Precision & Recall:** Critical for understanding the model's performance on the minority class (e.g., correctly identifying promoted employees).

* **F1-Score:** The harmonic mean of Precision and Recall.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def evaluate_model(model, X_test, y_test, model_name):
    # Get predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"--- {model_name} Performance ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nConfusion Matrix:")
    
    # Plot the matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()
    
    return accuracy, recall, f1

### A. Gaussian Naive Bayes 
We start with Gaussian Naive Bayes as our baseline. 

* **Why?** It is computationally efficient and provides a probabilistic baseline for comparison.

* **Assumption:** It assumes that the continuous features follow a Gaussian distribution (which we addressed via Log Transformation in Phase 3).

* **Evaluation Focus:** Since our target `promoted` is imbalanced, we prioritize **Recall** and **F1-Score** over simple accuracy to ensure we are correctly identifying the minority class of promoted employees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Initialize model
nb_model = GaussianNB(var_smoothing=1e-8)

# 2. Train model
nb_model.fit(X_train_resampled, y_train_resampled)

# 3. Get predicted probabilities
y_proba = nb_model.predict_proba(X_test)[:, 1]

# 4. Tune threshold
thresholds = np.arange(0.1, 0.9, 0.01)

best_f1 = 0
best_thresh = 0.5

for t in thresholds:
    y_pred_temp = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred_temp)
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Best Threshold: {best_thresh:.2f}")
print(f"Best F1-score: {best_f1:.4f}")

# 5. Final predictions
y_pred_optimized = (y_proba >= best_thresh).astype(int)

# 6. Classification report
print("\n--- Optimized Classification Report ---")
print(classification_report(y_test, y_pred_optimized))

# 3. Evaluate using your utility function
nb_acc, rb_recall, nb_f1 =evaluate_model(nb_model, X_test, y_test, "Naive Bayes")


### B. Random Forest:
* **Why?** It models non-linear interactions between variables, allowing it to capture complex relationships that a simple baseline might overlook.

* **Mechanism**: By building multiple decision trees during training and aggregating their results, the model effectively minimizes the risk of overfitting the training data.

* **Evaluation Focus**: Because it creates a robust decision boundary, we evaluate its success by how well it maintains a high F1-Score while handling the feature interactions inherent in our binned and encoded HR data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Initialize the Random Forest
rf_model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1) 


# 2. Train on the SMOTE-resampled training data 
rf_model.fit(X_train_resampled, y_train_resampled)
y_pred_rf= rf_model.predict(X_test)

print("--- Classification Report ---")
print(classification_report(y_test,y_pred_rf ))

# 3. Evaluate using your utility function
rf_acc, rf_recall, rf_f1 = evaluate_model(rf_model, X_test, y_test, "Random Forest")

### C. Neural Network Architecture
We implement a Multi-Layer Perceptron as our final, most sophisticated architecture for this analysis.

* **Why?** It is designed to identify high-dimensional, abstract patterns within the data that traditional algorithms may fail to detect.

* **Mechanism**: By utilizing multiple hidden layers and non-linear activation functions, the network learns to map complex input profiles to the probability of promotion.

* **Evaluation Focus**: We prioritize the model's ability to maximize its Recall, ensuring that it is not just learning common patterns, but is sensitive enough to consistently surface the most promising candidates from the minority class.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Initialize the MLPClassifier
nn_model = MLPClassifier( hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=500, random_state=42, )

# 2. Train on the SMOTE-resampled training data 
nn_model.fit(X_train_resampled, y_train_resampled)

# 4. Evaluate
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

nn_acc, nn_recall, nn_f1 = evaluate_model(nn_model, X_test, y_test, "Neural Network")

### 3. Final Model Performance Comparison
We consolidate our findings into a final comparative analysis to determine which architecture is most effective for the promotion prediction task.

* **Comparison Logic**: We evaluate the three models side-by-side using the results DataFrame to ensure that our selection is backed by hard, quantitative evidence.

* **Metric Synthesis**: We rely on the F1-Score as our primary tie-breaker, as it provides the necessary balance between Precision and Recall for an imbalanced dataset.

* **Goal**: To identify the "best fit" model that successfully captures the underlying patterns of employee promotion while maintaining generalizability to new data.

In [ ]:
import pandas as pd

# Assemble the captured metrics into a professional DataFrame
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Random Forest', 'Neural Network'],
    'Accuracy': [nb_acc, rf_acc, nn_acc],
    'Recall': [nb_recall, rf_recall, nn_recall],
    'F1-Score': [nb_f1, rf_f1, nn_f1]
})

# Display the table
print("--- Final Model Performance Comparison ---")
print(results.to_string(index=False))

# Identify the best model by F1-Score 
best_model_name = results.loc[results['F1-Score'].idxmax(), 'Model']
print(f"\nConclusion: The {best_model_name} is the best fit based on the highest F1-Score.")